# Orchestration & Automation with Databricks Workflows

Multi-task jobs, dependencies, scheduling, retries, and repair runs — the essentials of pipeline orchestration.

| Training Block | Duration | Type |
|---|---|---|
| Orchestration & Automation — Demo | 25 min | Demo |

**Prerequisites:** 03 — Lakeflow Pipelines (understanding of pipeline concepts)

## Learning Objectives

After completing this module you will be able to:

- **Create** multi-task Databricks Workflows with task dependencies
- **Configure** scheduling patterns (CRON expressions)
- **Apply** retry policies and understand repair runs
- **Use** `dbutils.widgets` and `dbutils.jobs.taskValues` for parameterization
- **Monitor** job execution via UI and system tables

## Introduction to Lakeflow Jobs

**Lakeflow Jobs** (formerly Databricks Jobs) is a managed orchestration service.

| Scenario | Solution |
|----------|----------|
| ETL pipeline with multiple steps | Multi-task Job |
| Daily report at fixed time | Scheduled Job |
| Reaction to new files | File Arrival Trigger |
| ML training pipeline | Job with notebook tasks |
| Run Lakeflow Pipeline | Job with Pipeline task |

| Feature | Jobs | Lakeflow Pipelines |
|---------|------|-----|
| Orchestration | General | ETL only |
| Dependencies | Manual configuration | Automatic (DAG) |
| Data Quality | Custom code | Built-in expectations |
| Flexibility | High | Opinionated |

**Best Practice**: Use Lakeflow Pipelines for ETL, Jobs for orchestrating Pipelines + other tasks.

---

### Task Types in Lakeflow Jobs

Overview of all available task types in Lakeflow Jobs and when to use each one.

| Task Type | Description | Use Case |
|-----------|-------------|----------|
| Notebook | Run a Databricks notebook | ETL logic, ML training |
| Pipeline | Run a Lakeflow Declarative Pipeline | Streaming/batch pipelines |
| Python Script | Run a Python file | Utility scripts |
| SQL | Run a SQL query | DDL, reporting queries |
| JAR | Run a Java/Scala JAR | Legacy Spark jobs |
| Spark Submit | Submit a Spark application | Custom Spark apps |
| If/Else Condition | Branch based on condition | Conditional workflows |
| For Each | Iterate over a list | Parameterized batch runs |

**Repair Runs:** Re-runs only **failed and downstream tasks**, skipping successful ones — saves compute and time.

**Pro Tip:** Know that repair runs skip already-successful tasks. If/Else and For Each enable conditional and iterative workflows.

## Preparing Notebooks for Job

Below are 3 simple notebooks that we'll use in the demo.

**Instructions**: 
1. Create folder `/Workspace/Users/<your-email>/jobs_demo/`
2. Copy each of the following code snippets to a separate notebook

---

### Task 1: Validate Source

Validates row count against `min_rows` threshold. Returns status + count via `dbutils.notebook.exit()`.

In [0]:
# TASK 1: Validate Source Data
# Copy this code to notebook: task_01_validate

# Parameters from Job
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("source_table", "bronze.bronze_orders", "Source Table")
dbutils.widgets.text("min_rows", "100")

catalog = dbutils.widgets.get("catalog")
source_table = dbutils.widgets.get("source_table")
min_rows = int(dbutils.widgets.get("min_rows"))

full_table = f"{catalog}.{source_table}" if catalog else source_table

# Validation
df = spark.table(full_table)
row_count = df.count()

if row_count < min_rows:
    raise Exception(f"Validation FAILED: {row_count} rows < {min_rows} minimum")

# Return result to next task
import json
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "source_table": full_table,
    "row_count": row_count
}))


### Task 2: Transform Data

Reads previous task result via `taskValues`, applies transformations (duration, cost per mile). Returns row count.

**Task Values — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Set value (upstream task) | `dbutils.jobs.taskValues.set(key="row_count", value=1000)` |
| Get value (downstream task) | `val = dbutils.jobs.taskValues.get(taskKey="Task_1", key="row_count")` |
| Supported types | `str`, `int`, `float`, `bool`, JSON-serializable `dict`/`list` |
| Max value size | 48 KB per key |


In [0]:
# TASK 2: Transform Data (Bronze → Silver)
# Copy this code to notebook: task_02_transform

from pyspark.sql.functions import *
import json
from datetime import date

# Parameters
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("source_table", "bronze.bronze_orders", "Source Table")
dbutils.widgets.text("run_date", "")

catalog = dbutils.widgets.get("catalog")
source_table = f'{catalog}.{dbutils.widgets.get("source_table")}' if catalog else dbutils.widgets.get("source_table")
run_date = dbutils.widgets.get("run_date") or str(date.today())

# Get result from previous task (optional)
try:
    prev_result = dbutils.jobs.taskValues.get(
        taskKey="validate",
        key="returnValue",
        default="{}"
    )
    prev_data = json.loads(prev_result)
    print(f"Previous task result: {prev_data}")
except:
    print("Running standalone (no previous task)")

# Transformation: compute net amounts
print(f"Transforming: {source_table}")

df = spark.table(source_table)

df_transformed = (
    df
    .withColumn("gross_amount", col("quantity") * col("unit_price"))
    .withColumn("discount_amount",
                round(col("quantity") * col("unit_price") * col("discount_percent") / 100, 2))
    .withColumn("net_amount",
                round(col("quantity") * col("unit_price") * (1 - col("discount_percent") / 100), 2))
    .withColumn("order_date", col("order_datetime").cast("date"))
    .withColumn("processing_date", lit(run_date))
    .filter(col("order_id").isNotNull())
    .filter(col("quantity") > 0)
)

row_count = df_transformed.count()
print(f"Transformed {row_count:,} rows")

df_transformed.select(
    "order_id", "customer_id", "quantity", "unit_price",
    "gross_amount", "discount_amount", "net_amount"
).show(5)

# Return result
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "rows_transformed": row_count
}))


### Task 3: Generate Report

Aggregates metrics (total trips, revenue, avg fare/distance). Prints summary report.

In [0]:
# TASK 3: Generate Sales Report
# Copy this code to notebook: task_03_report

from pyspark.sql.functions import *
import json
from datetime import datetime

# Parameters
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("source_table", "silver.silver_orders", "Source Table")

catalog = dbutils.widgets.get("catalog")
source_table = f'{catalog}.{dbutils.widgets.get("source_table")}' if catalog else dbutils.widgets.get("source_table")

# Aggregations
df = spark.table(source_table)

report = df.agg(
    count("*").alias("total_orders"),
    countDistinct("customer_id").alias("unique_customers"),
    round(sum("net_amount"), 2).alias("total_revenue"),
    round(avg("net_amount"), 2).alias("avg_order_value"),
    round(max("net_amount"), 2).alias("max_order_value"),
    round(avg("discount_percent"), 1).alias("avg_discount_pct")
).collect()[0]

# Display report
print("\n" + "=" * 50)
print("  RETAILHUB — DAILY SALES REPORT")
print("=" * 50)
print(f"  Total Orders:      {report.total_orders:,}")
print(f"  Unique Customers:  {report.unique_customers:,}")
print(f"  Total Revenue:     ${report.total_revenue:,.2f}")
print(f"  Avg Order Value:   ${report.avg_order_value:.2f}")
print(f"  Max Order Value:   ${report.max_order_value:.2f}")
print(f"  Avg Discount:      {report.avg_discount_pct}%")
print("=" * 50)
print(f"  Generated at:      {datetime.now()}")
print("=" * 50 + "\n")

# Return result
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "total_orders": report.total_orders,
    "total_revenue": float(report.total_revenue)
}))


### [UI DEMO] Creating Multi-task Job

**Step 1: Create new Job**
- [ ] Workflows → Create Job
- [ ] Name: `Demo_ETL_Pipeline`

<img src="../../../assets/images/93c107ca21a54aab98249cf47db0337d.png" width="800">


- [ ] Cluster Job: Create new cluster job

<img src="../../../assets/images/a967557a143a40c0ac7ed26ce469866a.png" width="800">

**Step 2: Add Task 1 (Validate)**
- [ ] Task name: `validate`
- [ ] Type: Notebook
- [ ] Path: `/Workspace/.../task_01_validate`
- [ ] Cluster: Serverless or new Job Cluster
- [ ] Parameters: `source_table` = `${CATALOG}.bronze.bronze_orders`

<img src="../../../assets/images/214b868309344df3a6e81f3cc2a84c13.png" width="800">

**Step 3: Add Task 2 (Transform)**
- [ ] Task name: `transform`
- [ ] Depends on: `validate`
- [ ] Path: `/Workspace/.../task_02_transform`
- [ ] Parameters: `source_table` = `${CATALOG}.bronze.bronze_orders`

**Step 4: Add Task 3 (Report)**
- [ ] Task name: `report`
- [ ] Depends on: `transform`
- [ ] Path: `/Workspace/.../task_03_report`

<img src="../../../assets/images/a3cc387f44de4247bf275bdbd38efb84.png" width="800">

**Step 5: Run Job**
- [ ] Run now
- [ ] Show: DAG visualization
- [ ] Show: Task logs and output

---

## [UI DEMO 2] Medallion Pipeline Job — Ready-to-Use Config

We already have 6 production-ready notebooks in `materials/medallion/` that implement a full Medallion pipeline:

| Layer | Notebook | Input | Output |
|-------|----------|-------|--------|
| **Bronze** | `bronze_customers` | CSV files | `bronze.bronze_customers` |
| **Bronze** | `bronze_orders` | JSON files | `bronze.bronze_orders` |
| **Silver** | `silver_customers` | bronze_customers | `silver.silver_customers` |
| **Silver** | `silver_orders_cleaned` | bronze_orders | `silver.silver_orders_cleaned` |
| **Gold** | `gold_customer_orders_summary` | silver_customers + silver_orders | `gold.gold_customer_orders_summary` |
| **Gold** | `gold_daily_orders` | silver_orders | `gold.gold_daily_orders` |

**DAG Structure:**
```
bronze_customers ──→ silver_customers ──────→ gold_customer_orders_summary
bronze_orders ────→ silver_orders_cleaned ─┤
 └→ gold_daily_orders
```

**How to deploy:** Copy the JSON config below → Workflows → Create Job → switch to JSON editor (or use Databricks CLI: `databricks jobs create --json @job.json`).

---

### Declarative Automation Bundles (DABs) — YAML Configuration

Below is the **Declarative Automation Bundles** (formerly Databricks Asset Bundles / DABs) YAML definition for the Medallion pipeline Job. This was exported from a working Databricks deployment.

> **Before deploying:** Update `notebook_path`, `existing_cluster_id`, and `parameters` to match your environment.

```yaml
resources:
 jobs:
 job_medalion_run:
 name: job_medalion_run
 tasks:
 - task_key: bronze_customer
 notebook_task:
 notebook_path: /Workspace/Users/<your-email>/materials/medallion/bronze_customers
 source: WORKSPACE
 existing_cluster_id: <YOUR_CLUSTER_ID>

 - task_key: bronze_orders
 notebook_task:
 notebook_path: /Workspace/Users/<your-email>/materials/medallion/bronze_orders
 source: WORKSPACE
 existing_cluster_id: <YOUR_CLUSTER_ID>

 - task_key: silver_customers
 depends_on:
 - task_key: bronze_customer
 notebook_task:
 notebook_path: /Workspace/Users/<your-email>/materials/medallion/silver_customers
 source: WORKSPACE
 existing_cluster_id: <YOUR_CLUSTER_ID>

 - task_key: silver_orders_cleaned
 depends_on:
 - task_key: bronze_orders
 notebook_task:
 notebook_path: /Workspace/Users/<your-email>/materials/medallion/silver_orders_cleaned
 source: WORKSPACE
 existing_cluster_id: <YOUR_CLUSTER_ID>

 - task_key: gold_customer_order_summary
 depends_on:
 - task_key: silver_customers
 - task_key: silver_orders_cleaned
 notebook_task:
 notebook_path: /Workspace/Users/<your-email>/materials/medallion/gold_customer_orders_summary
 source: WORKSPACE
 existing_cluster_id: <YOUR_CLUSTER_ID>

 - task_key: gold_daily_orders
 depends_on:
 - task_key: silver_orders_cleaned
 notebook_task:
 notebook_path: /Workspace/Users/<your-email>/materials/medallion/gold_daily_orders
 source: WORKSPACE
 existing_cluster_id: <YOUR_CLUSTER_ID>

 queue:
 enabled: true
 parameters:
 - name: catalog
 default: <YOUR_CATALOG>
 - name: source_path
 default: /Volumes/<YOUR_CATALOG>/default/datasets
```

| Element | Description |
|---------|-------------|
| `existing_cluster_id` | Use existing all-purpose cluster (for demo) or replace with `job_cluster_key` for production |
| `depends_on` | Defines task dependencies — creates the DAG |
| `parameters` | Job-level parameters passed to all notebooks via `dbutils.widgets` |
| `queue.enabled` | Queues runs if max concurrent reached |
| `source: WORKSPACE` | Notebook is in Workspace (not Repos) |

### Deploy Checklist

**Option A — JSON Editor (UI):**
1. [ ] Workflows → Create Job
2. [ ] Click ` Edit as JSON` (top-right)
3. [ ] Paste the JSON above (replace placeholders)
4. [ ] Save → Run now

**Option B — Databricks CLI:**
```bash
# Save JSON to file, then:
databricks jobs create --json @medallion_pipeline_job.json
```

**After deployment — show participants:**
- [ ] DAG visualization (fan-out at Bronze, fan-in at Gold)
- [ ] Task-level parameters (catalog, schema, source_path)
- [ ] Trigger: set to PAUSED — run manually for demo
- [ ] Run → observe sequential layer execution (Bronze → Silver → Gold)
- [ ] Show Repair Run: intentionally fail one task → repair reruns only failed + downstream

---

## [UI DEMO] Triggers and Schedule

How to configure different trigger types for Lakeflow Jobs — scheduled (CRON), file arrival, continuous, and manual triggers.

---

| Trigger Type | Usage | Example |
|---|---|---|
| **Scheduled** | Fixed schedule (CRON) | `0 0 2 * * ?` — daily at 2:00 |
| **File arrival** | Reaction to new files | New file in UC Volume |
| **Continuous** | Continuous processing | Streaming-like |
| **Manual** | On-demand | Testing |

**Pro Tip:** Know CRON syntax and File Arrival trigger configuration.


### Trigger Configuration Checklist

Step-by-step instructor checklist for demonstrating trigger options in the Lakeflow Jobs UI.

**Trigger Options** (Triggers tab):

| Trigger Type | Usage | Example |
|--------------|-------|---------|
| **Scheduled** | Fixed schedule | Daily at 2:00 |
| **File arrival** | Reaction to new files | New file in `/landing/` |
| **Continuous** | Continuous processing | Streaming-like |
| **Manual** | On-demand | Testing |

<img src="../../../assets/images/6e23746ce063491fa8afb3dea6268a1d.png" width="800">


**Demo: Scheduled Trigger**
- [ ] Add trigger → Scheduled
- [ ] Cron expression: `0 0 2 * * ?` (daily at 2:00)
- [ ] Timezone: `Europe/Warsaw`
- [ ] Show: Preview next runs


<img src="../../../assets/images/cf3cfc85162a466eb77e15e20df5c15c.png" width="800">

**Demo: File Arrival Trigger** (optional)
- [ ] Add trigger → File arrival
- [ ] URL: Unity Catalog Volume path
- [ ] Min files: 1
### Useful CRON Expressions

Common CRON patterns for scheduling Lakeflow Jobs at various intervals.

```
0 0 2 * * ? # Daily at 2:00
0 0 * * * ? # Every hour
0 0 9 ? * MON-FRI # Mon-Fri at 9:00
0 0 0 1 * ? # First day of month
0 */15 * * * ? # Every 15 minutes
```

---

## [UI DEMO] Options, Retry and Alerting

**Task-level:** Timeout, Retries (count + delay) 
**Job-level:** Max concurrent runs, Job timeout 
**Notifications:** Email (on failure/success), Webhooks (Slack/Teams via Destinations)

| Scenario | Retry? | Why |
|----------|--------|-----|
| Network timeout | Yes | Transient error |
| API rate limit | Yes | Transient error |
| Data quality issue | No | Retry won't fix data |
| Code bug | No | Retry won't fix code |

---

## Demo: Widgets and Parameters

Databricks Widgets allow you to parameterize notebooks.

---

**Databricks Widgets — Syntax Reference**

| Widget Type | Syntax |
|-------------|--------|
| Text input | `dbutils.widgets.text("name", "default", "Label")` |
| Dropdown | `dbutils.widgets.dropdown("name", "default", ["opt1", "opt2"], "Label")` |
| Multi-select | `dbutils.widgets.multiselect("name", "default", ["opt1", "opt2"], "Label")` |
| Combobox | `dbutils.widgets.combobox("name", "default", ["opt1", "opt2"], "Label")` |
| Get value | `val = dbutils.widgets.get("name")` |
| Remove all | `dbutils.widgets.removeAll()` |


In [0]:
# Widget types

# Text - any text
dbutils.widgets.text("environment", "dev", "Environment")

# Dropdown - select from list
dbutils.widgets.dropdown("region", "EU", ["EU", "US", "APAC"], "Region")

# Combobox - dropdown with typing option
dbutils.widgets.combobox("table", "orders", ["orders", "customers", "products"], "Table")

# Multiselect - multiple selection
dbutils.widgets.multiselect("columns", "id", ["id", "name", "date", "amount"], "Columns")

In [0]:
# Getting values
environment = dbutils.widgets.get("environment")
region = dbutils.widgets.get("region")
table = dbutils.widgets.get("table")
columns = dbutils.widgets.get("columns")  # returns comma-separated string

print(f"Environment: {environment}")
print(f"Region: {region}")
print(f"Table: {table}")
print(f"Columns: {columns}")

In [0]:
# Dynamic parameters in Job
# These values are available when notebook is run as a task in Job

dynamic_params = {
    "{{job.start_time.iso_date}}": "Run date (YYYY-MM-DD)",
    "{{job.start_time}}": "Full timestamp",
    "{{job.id}}": "Job ID",
    "{{run.id}}": "Current run ID",
    "{{task.name}}": "Current task name"
}

for param, description in dynamic_params.items():
    print(f"{param:35} -> {description}")

In [0]:
# Widget cleanup
dbutils.widgets.removeAll()

### Orchestrating Notebooks — dbutils.notebook.run()

While Lakeflow Jobs is the production-grade orchestrator, `dbutils.notebook.run()` lets you call one notebook from another programmatically — useful for lightweight pipelines and testing.

```python
result = dbutils.notebook.run(
    path             = "/path/to/child_notebook",
    timeout_seconds  = 300,
    arguments        = {"env": "prod", "min_rows": "500"}
)
print(result)   # child calls dbutils.notebook.exit("OK: 1234 rows")
```

#### Task Values — passing data between Job tasks

In a multi-task Job you can set and retrieve task-scoped values:

```python
# In task A — set a value
dbutils.jobs.taskValues.set("row_count", 1234)

# In task B (downstream) — read the value
count = dbutils.jobs.taskValues.get("task_A", "row_count", default=0)
```

| Method | Scope | Use Case |
|---|---|---|
| `dbutils.notebook.run()` | Notebook-to-notebook | Quick orchestration, child notebook pattern |
| `dbutils.jobs.taskValues` | Task-to-task in a Job | Pass computed values between Job tasks |

> **Pro Tip:** `dbutils.notebook.run()` is synchronous — it blocks until the child finishes or the timeout is reached. For parallel execution use Lakeflow Jobs task dependencies.

---

In [ ]:
# dbutils.notebook.run() — programmatic notebook orchestration
# (Uncomment to use in production; the path below is illustrative)

# result = dbutils.notebook.run(
#     "/path/to/validation_notebook",
#     timeout_seconds = 120,
#     arguments       = {
#         "env"     : dbutils.widgets.get("environment"),
#         "min_rows": "100"
#     }
# )
# print(f"Child notebook returned: {result}")

# ── dbutils.jobs.taskValues — passing data between Job tasks ──────────────
# In Task A:
# dbutils.jobs.taskValues.set("validated_rows", 9999)

# In Task B:
# row_count = dbutils.jobs.taskValues.get("task_a", "validated_rows", default=0)

# Signature check
print("dbutils.notebook.run  →", dbutils.notebook.run.__doc__[:80] if hasattr(dbutils.notebook.run,'__doc__') and dbutils.notebook.run.__doc__ else "available in Databricks runtime")
print("dbutils.jobs.taskValues →", "set(key, value) / get(taskKey, key, default)")

## 🎁 Bonus: Monitoring via System Tables

Key table: `system.lakeflow.job_run_timeline` — contains run history, duration, result state.

---

In [0]:
spark.sql("""
    SELECT 
        *
    FROM system.lakeflow.job_run_timeline
    ORDER BY 1 DESC
    LIMIT 20
""").display()

In [0]:
spark.sql("""
    SELECT 
        DATE(period_start_time) as run_date,
        run_name as job_name,
        ROUND(
            AVG(
                (UNIX_TIMESTAMP(period_end_time) - UNIX_TIMESTAMP(period_start_time)) / 60
            ), 1
        ) as avg_duration_min,
        COUNT(*) as runs
    FROM system.lakeflow.job_run_timeline
    WHERE period_start_time >= current_date() - INTERVAL 14 DAYS
        AND result_state = 'SUCCESS'
    GROUP BY run_date, job_name
    ORDER BY run_date DESC
""").display()

## Serverless Compute for Jobs

Since 2025, Databricks offers **Serverless compute** as the default option for Lakeflow Jobs:

| Aspect | Classic Clusters | Serverless Compute |
|--------|-----------------|-------------------|
| **Startup time** | 3-10 minutes | Seconds |
| **Cluster management** | Manual sizing | Automatic |
| **Cost model** | Per-VM (DBU + cloud) | Per-DBU only |
| **Spot instances** | Configurable | Not applicable |
| **Use case** | Long-running, GPU, custom libs | ETL, SQL, notebooks |

**How to enable:** In the Job task configuration, select **Serverless** as the compute type instead of specifying a cluster.

> **Pro Tip:** Serverless compute eliminates cold start delays — ideal for triggered jobs that need fast response times. For cost-sensitive batch jobs running on schedule, classic clusters with spot instances may still be cheaper.

---

## Summary

| Topic | Key Concept | Key Terms |
|---|---|---|
| **Multi-task Jobs** | DAG workflow, task dependencies | Task types, Repair Runs |
| **Triggers** | Scheduled (CRON), File arrival, Continuous | `0 0 2 * * ?` |
| **Options** | Timeout, Retry, Max concurrent runs | Transient vs permanent errors |
| **Alerting** | Email, Webhooks (Slack/Teams) | Notification destinations |
| **Parameters** | Widgets, dynamic values, taskValues | `dbutils.widgets`, `dbutils.jobs.taskValues` |
| **Monitoring** | System tables, success rate, duration | `system.lakeflow.job_run_timeline` |

---

← [03 — Lakeflow Pipelines](03_lakeflow_pipelines_demo.ipynb) | **[ README](../../../README.md)**
